# 分箱方法示例 - 分别导入各分箱类

本示例展示 hscredit 中各分箱方法的**分别导入**使用方式。
如需统一接口，请参考 `03_optimal_binning_unified.ipynb`。

## 1. 环境准备

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# 添加项目路径
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 分别导入各分箱类（旧方式）
from hscredit.core.binning import (
    UniformBinning,      # 等宽分箱
    QuantileBinning,     # 等频分箱
    TreeBinning,         # 决策树分箱
    ChiMergeBinning,     # 卡方分箱
    OptimalKSBinning,    # 最优KS分箱
    OptimalIVBinning,    # 最优IV分箱
    MDLPBinning,         # MDLP分箱
    CartBinning,         # CART分箱
    MonotonicBinning,    # 单调性约束分箱
    GeneticBinning,      # 遗传算法分箱
    SmoothBinning,       # 平滑分箱
    KernelDensityBinning,# 核密度分箱
    BestLiftBinning,     # Best Lift分箱
    TargetBadRateBinning,# 目标坏样本率分箱
    KMeansBinning,       # K-Means分箱
)

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 分箱类导入成功（分别导入方式）")

✅ 分箱类导入成功（分别导入方式）


## 2. 加载数据

In [2]:
# 加载示例数据
data_path = Path("../utils/hscredit.xlsx")
df = pd.read_excel(data_path)

print(f"数据集形状: {df.shape}")
print(f"\n列名:\n{df.columns.tolist()}")
print(f"\n数据类型:\n{df.dtypes}")
print(f"\n前5行:\n{df.head()}")

数据集形状: (22729, 12)

列名:
['MOB1', 'MOB2', '青云24', '游昆定制分80', '百融定制分V9', '中智小牛分C3', '度小满欺诈因子V6PRO多头版', '百行百川分FPV1', '设备黑名单', '放款期数']

数据类型:
MOB1                 int64
MOB2                 int64
青云24               float64
游昆定制分80            float64
百融定制分V9            float64
中智小牛分C3            float64
度小满欺诈因子V6PRO多头版    float64
百行百川分FPV1          float64
设备黑名单            float64
放款期数                 int64
dtype: object

前5行:
   MOB1  MOB2   青云24  游昆定制分80     百融定制分V9  中智小牛分C3  度小满欺诈因子V6PRO多头版  \
0     0     0  612.0    718.0  914.739990    687.0              NaN   
1     0     0  640.0    718.0  894.280029    774.0              NaN   
2     0     0  581.0    709.0  822.119995    629.0        53.380001   
3     0     0  650.0    718.0  794.500000    662.0        48.650002   
4     0     0  650.0    718.0  794.500000    662.0        48.650002   

   百行百川分FPV1  设备黑名单   设备黑名单  放款期数  放款期数  
0        NaN      2.0  STORE      LYX_DX     6  
1        NaN      3.0  STORE  RENPIN_XXK     6  
2      

In [3]:
# 选择数值型特征作为示例
# 使用 MOB1 作为目标变量（0/1分类）
from numpy import int0


numeric_features = [
    '青云24', '游昆定制分80', '百融定制分V9', '中智小牛分C3',
    '度小满欺诈因子V6PRO多头版', '百行百川分FPV1', '设备黑名单'
]

X = df[numeric_features].copy()
y = (df['MOB1'] > 15).astype(int).copy()

# 处理缺失值（简单填充）
X = X.fillna(X.median())

print(f"特征数据: {X.shape}")
print(f"目标变量分布:\n{y.value_counts()}")
print(f"坏样本率: {y.mean():.2%}")

特征数据: (22729, 7)
目标变量分布:
MOB1
0    21122
1     1607
Name: count, dtype: int64
坏样本率: 7.07%


## 3. 基础分箱方法

In [4]:
# 3.1 等宽分箱
feature = '青云24'
uniform_binner = UniformBinning(max_n_bins=5)
uniform_binner.fit(X[[feature]], y)

print(f"【{feature}】等宽分箱结果:")
print(uniform_binner.get_bin_table(feature)[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【青云24】等宽分箱结果:
 分箱  样本总数     坏样本率
  0  3207 0.086997
  1  9402 0.079558
  2  8030 0.062017
  3  1939 0.040227
  4   151 0.026490


In [5]:
# 3.2 等频分箱
quantile_binner = QuantileBinning(max_n_bins=5)
quantile_binner.fit(X[[feature]], y)

print(f"【{feature}】等频分箱结果:")
print(quantile_binner.get_bin_table(feature)[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【青云24】等频分箱结果:
 分箱  样本总数     坏样本率
  0  4533 0.084933
  1  4534 0.083150
  2  4529 0.075072
  3  4558 0.060992
  4  4575 0.049617


In [6]:
# 3.3 决策树分箱
tree_binner = TreeBinning(max_n_bins=5)
tree_binner.fit(X[[feature]], y)

print(f"【{feature}】决策树分箱结果:")
print(tree_binner.get_bin_table(feature)[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【青云24】决策树分箱结果:
 分箱  样本总数     坏样本率
  0  4258 0.085486
  1  1355 0.104059
  2  7983 0.074784
  3  7164 0.060581
  4  1969 0.036059


## 4. 优化分箱方法

In [21]:
# 4.1 卡方分箱
chi_binner = ChiMergeBinning(max_n_bins=5)
chi_binner.fit(X[[feature]], y)

print(f"【{feature}】卡方分箱结果:")
print(chi_binner.get_bin_table(feature)[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【百融定制分V9】卡方分箱结果:
 分箱  样本总数     坏样本率
  0 18947 0.078588
  1   828 0.031401
  2   272 0.069853
  3  1629 0.033763
  4  1053 0.017094


In [22]:
# 4.2 最优KS分箱
ks_binner = OptimalKSBinning(max_n_bins=5)
ks_binner.fit(X[[feature]], y)

print(f"【{feature}】最优KS分箱结果:")
print(ks_binner.get_bin_table(feature)[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【百融定制分V9】最优KS分箱结果:
 分箱  样本总数     坏样本率
  0   133 0.157895
  1   158 0.113924
  2   146 0.143836
  3 10321 0.093111
  4 11971 0.048952


In [23]:
# 4.3 最优IV分箱（推荐）
iv_binner = OptimalIVBinning(max_n_bins=5)
iv_binner.fit(X[[feature]], y)

print(f"【{feature}】最优IV分箱结果:")
print(iv_binner.get_bin_table(feature)[['分箱', '样本总数', '坏样本率', '分档IV值']].to_string(index=False))

【百融定制分V9】最优IV分箱结果:
 分箱  样本总数     坏样本率    分档IV值
  0  3840 0.113542 0.057377
  1  9510 0.079916 0.007771
  2  5597 0.052349 0.022018
  3  2729 0.036643 0.043147
  4  1053 0.017094 0.055787


In [24]:
feature = "百融定制分V9"
# 4.4 MDLP分箱（自动确定分箱数）
mdlp_binner = MDLPBinning(max_n_bins=5)
mdlp_binner.fit(X[[feature]], y)

print(f"【{feature}】MDLP自动分箱数: {mdlp_binner.n_bins_[feature]}")
print("MDLP分箱结果:")
print(mdlp_binner.get_bin_table(feature)[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【百融定制分V9】MDLP自动分箱数: 4
MDLP分箱结果:
 分箱  样本总数     坏样本率
  0  3737 0.114798
  1  7746 0.082623
  2  8715 0.054274
  3  2531 0.025682


## 5. 高级分箱方法

In [25]:
# 5.1 CART分箱
cart_binner = CartBinning(max_n_bins=5)
cart_binner.fit(X[[feature]], y)

print(f"【{feature}】CART分箱结果:")
print(cart_binner.get_bin_table(feature)[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【百融定制分V9】CART分箱结果:
 分箱  样本总数     坏样本率
  0  3737 0.114798
  1  6808 0.085194
  2  5026 0.064465
  3  4619 0.045248
  4  2539 0.025601


In [26]:
# 5.2 单调性约束分箱
# monotonic参数支持多种选项：
# - False: 不要求单调性
# - True 或 'auto': 自动检测最佳单调方向
# - 'ascending': 强制坏样本率递增
# - 'descending': 强制坏样本率递减
# - 'peak': 允许单峰形态（先升后降）
# - 'valley': 允许单谷形态（先降后升）
mono_binner = MonotonicBinning(
    monotonic='descending',   # 强制递减
    max_n_bins=5
)
mono_binner.fit(X[[feature]], y)

print(f"【{feature}】单调递减分箱结果:")
print(mono_binner.get_bin_table(feature)[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【百融定制分V9】单调递减分箱结果:
 分箱  样本总数     坏样本率
  0  3409 0.113816
  1  5683 0.087982
  2  4545 0.071507
  3  7955 0.046889
  4  1137 0.018470


In [27]:
# 对比不同单调性选项
print("单调性参数对比:")
print("="*50)
print(f"{'参数':<12} {'说明':<20} {'分箱数':>10}")
print("-"*50)

monotonic_options = {
    'auto': '自动检测单调方向',
    'ascending': '强制坏样本率递增',
    'descending': '强制坏样本率递减',
    'peak': '允许单峰形态(先升后降)',
    'valley': '允许单谷形态(先降后升)'
}

for option, desc in monotonic_options.items():
    binner = MonotonicBinning(monotonic=option, max_n_bins=5)
    binner.fit(X[[feature]], y)
    n_bins = binner.n_bins_[feature]
    print(f"{option:<12} {desc:<20} {n_bins:>10}")

单调性参数对比:
参数           说明                          分箱数
--------------------------------------------------
auto         自动检测单调方向                      5
ascending    强制坏样本率递增                      2
descending   强制坏样本率递减                      5
peak         允许单峰形态(先升后降)                  5
valley       允许单谷形态(先降后升)                  5


## 6. 高级分箱方法（续）

In [28]:
# 6.1 遗传算法分箱
genetic_binner = GeneticBinning(max_n_bins=5)
genetic_binner.fit(X[[feature]], y)

print(f"【{feature}】遗传算法分箱结果:")
print(genetic_binner.get_bin_table(feature)[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【百融定制分V9】遗传算法分箱结果:
 分箱  样本总数     坏样本率
  0  7576 0.101637
  1  8022 0.070307
  2  3563 0.044906
  3  2231 0.038100
  4  1337 0.020942


In [29]:
# 6.2 平滑分箱
smooth_binner = SmoothBinning(max_n_bins=5, n_prebins=50)
smooth_binner.fit(X[[feature]], y)

print(f"【{feature}】平滑分箱结果:")
print(smooth_binner.get_bin_table(feature)[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【百融定制分V9】平滑分箱结果:
 分箱  样本总数     坏样本率
  0 10606 0.095323
  1 12123 0.049163


In [30]:
# 6.3 核密度分箱
kd_binner = KernelDensityBinning(max_n_bins=5)
kd_binner.fit(X[[feature]], y)

print(f"【{feature}】核密度分箱结果:")
print(kd_binner.get_bin_table(feature)[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【百融定制分V9】核密度分箱结果:
 分箱  样本总数     坏样本率
  0 13451 0.089287
  1  9278 0.043759


In [31]:
# 6.4 Best Lift分箱
# 新算法：预分割后贪心合并，优化头部/尾部的极端Lift值
from pandas.core.frame import DataFrame


lift_binner = BestLiftBinning(
    max_n_bins=5,
    min_lift=0.0,              # 默认值：不限制Lift阈值
    monotonic=True,            # 默认值：自动检测单调方向
    optimization='extreme',    # 优化目标：最大化极端箱的Lift
    n_prebins=50               # 预分箱数量
)
lift_binner.fit(X[[feature]], y)

print(f"【{feature}】Best Lift分箱结果:")
bin_table: DataFrame = lift_binner.get_bin_table(feature)
print(bin_table[['分箱', '样本总数', '坏样本率', 'LIFT值']].to_string(index=False))

【百融定制分V9】Best Lift分箱结果:
 分箱  样本总数     坏样本率    LIFT值
  0   454 0.134361 1.900372
  1  3181 0.111600 1.578446
  2 10002 0.079684 1.127033
  3  7727 0.047366 0.669934
  4  1365 0.020513 0.290131


In [32]:
# 6.6 K-Means分箱
# 使用scipy的kmeans2实现，避免macOS兼容性问题
kmeans_binner = KMeansBinning(
    max_n_bins=5,
    random_state=42,
    n_init=10          # K-Means初始化次数
)
kmeans_binner.fit(X[[feature]], y)

print(f"【{feature}】K-Means分箱结果:")
print(kmeans_binner.get_bin_table(feature)[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【百融定制分V9】K-Means分箱结果:
 分箱  样本总数     坏样本率
  0  2197 0.113336
  1  4111 0.097543
  2  5663 0.078404
  3  6450 0.056434
  4  4308 0.034587


## 7. 方法对比

In [33]:
# 对比不同方法的IV值
methods = {
    'Uniform': UniformBinning(max_n_bins=5),
    'Quantile': QuantileBinning(max_n_bins=5),
    'Tree': TreeBinning(max_n_bins=5),
    'ChiMerge': ChiMergeBinning(max_n_bins=5),
    'OptimalKS': OptimalKSBinning(max_n_bins=5),
    'OptimalIV': OptimalIVBinning(max_n_bins=5),
    'CART': CartBinning(max_n_bins=5),
    'MDLP': MDLPBinning(max_n_bins=5),
    'Monotonic': MonotonicBinning(max_n_bins=5),
    'Genetic': GeneticBinning(max_n_bins=5),
    'Smooth': SmoothBinning(max_n_bins=5),
    'KernelDensity': KernelDensityBinning(max_n_bins=5),
    'BestLift': BestLiftBinning(max_n_bins=5, optimization='extreme', n_prebins=50),
    'TargetBadRate': TargetBadRateBinning(max_n_bins=5, strict_mode=False),
    'KMeans': KMeansBinning(max_n_bins=5),
}

print(f"【{feature}】各方法IV值对比:")
print("="*50)
print(f"{'方法':<15} {'分箱数':>8} {'IV值':>10}")
print("-"*50)

for name, binner in methods.items():
    binner.fit(X[[feature]], y)
    table = binner.get_bin_table(feature)
    n_bins = len([b for b in table['分箱'] if b != 'missing'])
    iv = table['分档IV值'].sum()
    print(f"{name:<15} {n_bins:>8} {iv:>10.4f}")

【百融定制分V9】各方法IV值对比:
方法                   分箱数        IV值
--------------------------------------------------
Uniform                5     0.1641
Quantile               5     0.1666
Tree                   5     0.1931
ChiMerge               5     0.1172
OptimalKS              5     0.1296
OptimalIV              5     0.1861
CART                   5     0.1931
MDLP                   4     0.1772
Monotonic              5     0.1780
Genetic                5     0.1786
Smooth                 2     0.1245
KernelDensity          2     0.1275
BestLift               5     0.1758
TargetBadRate          5     0.0823
KMeans                 5     0.1550


## 7. WOE转换

In [34]:
# 使用最优IV分箱进行WOE转换
# 注意：min_bin_size 默认值为 0.01 (1%)，更灵活
binner = OptimalIVBinning(
    max_n_bins=5,
    min_bin_size=0.01,      # 每箱最小样本占比1%
    monotonic='auto'        # 自动检测单调方向
)
binner.fit(X, y)

# WOE转换
X_woe = binner.transform(X, metric='woe')

print("WOE转换结果:")
print(X_woe.head())

WOE转换结果:
       青云24   游昆定制分80   百融定制分V9   中智小牛分C3  度小满欺诈因子V6PRO多头版  百行百川分FPV1  设备黑名单
0  0.062846 -0.271842 -1.475839 -0.543717         0.066322   0.085155      0.0
1 -0.171444 -0.271842 -1.475839 -0.543717         0.066322   0.085155      0.0
2  0.062846 -0.050637 -0.320098  0.016301         0.066322  -1.504846      0.0
3 -0.171444 -0.271842 -0.320098  0.016301        -0.361079  -0.866073      0.0
4 -0.171444 -0.271842 -0.320098  0.016301        -0.361079  -0.866073      0.0


## 总结

本示例展示了分别导入各分箱类的使用方式。

**关键改进**：
1. **min_bin_size 默认改为 1%** (原 5%)，更灵活
2. **monotonic 参数增强**：支持 'auto', 'ascending', 'descending', 'peak', 'valley'
3. **BestLiftBinning 重写**：新算法优化头部/尾部极端Lift值
4. **TargetBadRateBinning 双模式**：严格边界模式 + 自动优化模式
5. **KMeansBinning 兼容性**：使用 scipy.kmeans2，避免 macOS 兼容性问题

**推荐使用统一接口 `OptimalBinning`**，详见 `03_optimal_binning_unified.ipynb`：
```python
from hscredit.core.binning import OptimalBinning

# 一行代码切换不同方法
binner = OptimalBinning(method='optimal_iv', max_n_bins=5)
binner.fit(X, y)
```